## Path/Import Cell

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## UI Function Cell

In [2]:
from intuition_engine.examples import EXAMPLES
from intuition_engine.pipeline import analyze_equation

def launch_engine_ui():

    import ipywidgets as widgets
    import sympy as sp
    from IPython.display import display, Markdown, clear_output, Math

    example_dropdown = widgets.Dropdown(
        options=list(EXAMPLES.keys()),
        value="Driven damped oscillator",
        description="Example:",
        layout=widgets.Layout(width="600px")
    )

    use_custom_checkbox = widgets.Checkbox(
        value=False,
        description="Use custom equation"
    )

    custom_input = widgets.Textarea(
        value=EXAMPLES["Driven damped oscillator"],
        description="Equation:",
        layout=widgets.Layout(width="900px", height="120px")
    )

    analyze_button = widgets.Button(
        description="Analyze Equation",
        button_style="success"
    )

    output_area = widgets.Output()

    def safe_latex(expr):
        if expr is None:
            return r"\text{None}"
        try:
            return sp.latex(expr)
        except Exception:
            return str(expr)

    def render_report(report):
        parsed = report.parsed
        classification = report.classification
        features = report.features
        scaling = report.scaling
        regimes = report.regimes
        validation = report.validation

        display(Markdown("## Input / Canonical Form"))
        display(Math(rf"{safe_latex(parsed.lhs)} = {safe_latex(parsed.rhs)}"))
        display(Math(rf"{safe_latex(parsed.canonical_expr)} = 0"))

        display(Markdown("## Validation"))

        if validation.is_supported:
            display(Markdown("- ✅ This equation is fully supported by the current v1 engine."))
        else:
            display(Markdown("- ⚠️ This equation is only partially supported by the current v1 engine."))
            display(Markdown("**Warnings**"))
            for w in validation.warnings:
                display(Markdown(f"- {w}"))

        display(Markdown("## Classification"))
        display(Markdown(
            f"""
            - **Order:** {classification.order}
            - **Linear:** {classification.is_linear}
            - **Constant coefficients:** {classification.has_constant_coefficients}
            - **Forced:** {classification.is_forced}
            - **Recognized family:** `{classification.family}`
            """
        ))

        if classification.notes:
            display(Markdown("**Notes**"))
            for note in classification.notes:
                display(Markdown(f"- {note}"))

        display(Markdown("## Extracted Features"))
        display(Math(rf"a = {safe_latex(features.a)}"))
        display(Math(rf"b = {safe_latex(features.b)}"))
        display(Math(rf"c = {safe_latex(features.c)}"))
        display(Math(rf"f(t) = {safe_latex(features.forcing)}"))
        display(Math(rf"\Delta = {safe_latex(features.discriminant)}"))
        display(Math(rf"\omega_n = {safe_latex(features.natural_frequency)}"))
        display(Math(rf"\zeta = {safe_latex(features.damping_ratio)}"))
        display(Math(rf"\tau_d \sim {safe_latex(features.damping_timescale)}"))
        display(Math(rf"\text{{forcing frequency}} = {safe_latex(features.forcing_frequency)}"))
        display(Math(rf"\text{{Characteristic polynomial: }} {safe_latex(features.characteristic_polynomial)}"))

        display(Markdown("## Scaling / Dimensionless Structure"))

        display(Math(rf"\omega_n = {safe_latex(scaling.natural_frequency)}"))
        display(Math(rf"\zeta = {safe_latex(scaling.damping_ratio)}"))
        
        if scaling.forcing_ratio is not None:
            display(Math(rf"r = \frac{{\omega}}{{\omega_n}} = {safe_latex(scaling.forcing_ratio)}"))
        
        if scaling.notes:
            for note in scaling.notes:
                display(Markdown(f"- {note}"))

        if features.roots is not None:
            display(Markdown("**Characteristic roots**"))
            for i, root in enumerate(features.roots, start=1):
                display(Math(rf"r_{i} = {safe_latex(root)}"))

        display(Markdown("## Regime Insights"))
        for r in regimes:
            display(Markdown(f"**{r.label}**"))
            try:
                display(Math(rf"{safe_latex(r.condition)}"))
            except Exception:
                display(Markdown(f"- Condition: `{r.condition}`"))
            display(Markdown(f"- Meaning: {r.meaning}"))
            
    def on_example_change(change):
        custom_input.value = EXAMPLES[change["new"]]

    def on_click(_):
        with output_area:
            clear_output()
            try:
                eq = custom_input.value.strip() if use_custom_checkbox.value else EXAMPLES[example_dropdown.value]
                report = analyze_equation(eq)
                render_report(report)
            except Exception as e:
                display(Markdown("## Error"))
                print(type(e).__name__ + ":", e)

    example_dropdown.observe(on_example_change, names="value")
    analyze_button.on_click(on_click)

    display(
        widgets.VBox([
            widgets.HTML("<h1>Physics Intuition Engine v1</h1>"),
            widgets.HTML("<p>Analyze second-order linear constant-coefficient ODEs and render extracted structure in LaTeX.</p>"),
            example_dropdown,
            use_custom_checkbox,
            custom_input,
            analyze_button,
            output_area
        ])
    )

## `launch_engine_ui()` Cell

In [3]:
launch_engine_ui()